In [11]:
import os
import io
import time
import uuid
from datetime import date

import pandas as pd
import requests
import sqlalchemy as sa
from google.cloud import storage
from google.oauth2 import service_account

PROJECT = "cbs-pratice"
BUCKET = "cbs-phuc-bucket-1"
API_URL = "https://xqqpbuhqvbbgyevwmktp.supabase.co/functions/v1"
SRC_BUCKET = "cbs-phuc-bucket-1"
LOCATION = "asia-southeast1"

DB_USER = os.environ.get('DB_USER', 'student_readonly')
DB_PWD = os.environ.get('DB_PWD', '7AhQE2LdyM5UV+PtkrwpzbKq+MW5LzkdDJwFX2HF')
DB_HOST = os.environ.get('DB_HOST', 'db.stylehub.delabs.io.vn')
DB_PORT = os.environ.get('DB_PORT', 5432)
DB_NAME = os.environ.get('DB_NAME', 'stylehub')
DB_URL = f"postgresql+psycopg2://{DB_USER}:{DB_PWD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Load credentials từ file JSON thành object trước
credentials = service_account.Credentials.from_service_account_file(
    'C:/CyberSoftDE/Practice/secrets/cedar-turbine-511010-d3-b318806c7288.json'
)
gcs_client = storage.Client(
    project=PROJECT,
    credentials=credentials
)

bucket = gcs_client.bucket(BUCKET)
if not bucket.exists():
    bucket = gcs_client.create_bucket(BUCKET, location=LOCATION)
    print(f"✅ Created bucket: gs://{BUCKET}")
else:
    print(f"✅ Bucket exists: gs://{BUCKET}")

✅ Bucket exists: gs://cbs-phuc-bucket-1


In [12]:
WATERMARK_COL = "updated_at"
engine = sa.create_engine(DB_URL)

def extract_day(logical_date):
    start = pd.Timestamp(logical_date)
    return pd.read_sql(
        sa.text(f"""select * from order_items where {WATERMARK_COL} >= :s and {WATERMARK_COL} < :e"""),
        engine,
        params={
            "s": start.to_pydatetime(),
            "e": (start + pd.Timedelta(days=1)).to_pydatetime()
        }
    )

def write_raw(df, table, source, logical_date, overwrite=False):
    batch_id = uuid.uuid4().hex[:8]
    df.assign(
        _ingested=pd.Timestamp(logical_date),
        _source=source,
        _batch_id=batch_id
    )
    if overwrite:
        path = f"raw/stylehub/{table}/date={logical_date}/order_items.parquet"
    else:
        path = f"raw/stylehub/{table}/date={logical_date}/batch_id={batch_id}/order_items.parquet"
    bucket.blob(path).upload_from_string(df.to_parquet(index=False))
    return path

def count_distinct_id(path):
    parquet_bytes = bucket.blob(path).download_as_bytes()
    df = pd.read_parquet(io.BytesIO(parquet_bytes))
    return df["id"].nunique()

def inges_day(logical_date):
    df = extract_day(logical_date)
    path = write_raw(df, "order_items", "stylehub:order_items", logical_date)
    return count_distinct_id(path)

In [14]:
DAYS = ["2026-10-01", "2026-10-02", "2026-10-03", "2026-10-04", "2026-10-05", "2026-10-06", "2026-10-07"]

distinct_id_count_dict = { }

for run_count in range(2):
    arr = []
    for d in DAYS:
        distinct_id_count = inges_day(d)
        arr.append(distinct_id_count)
    distinct_id_count_dict[run_count] = arr
    print(f"Run index {run_count + 1}: {distinct_id_count_dict[run_count]}")

Run index 1: [183260, 8935, 11004, 11981, 13228, 11051, 12951]
Run index 2: [183260, 8935, 11004, 11981, 13228, 11051, 12951]


1. Chọn watermark column là updated_at vì giá trị của updated_at chỉ thay đổi khi order_items thay đổi, không tự thay đổi khi dữ liệu của các bảng liên quan thay đổi nên chọn watermark column là updated_at sẽ giúp ghi nhận đúng thời gian thay đổi dữ liệu của order_items, tránh được việc trùng lặp dữ liệu khi query phải các bản ghi phù hợp về updated_at nhưng dữ liệu không thay đổi so với trước. Số ngày lookback là 1 để đảm bảo cập nhật đủ dữ liệu tránh thất thoát do lệch múi giờ hay lỗi mạng.
2. Khi chạy lại thì Raw Zone sẽ có thêm file order_items.parquet trong filer batch_id mới chứa dữ liệu order_items trong ngày được chọn và 1 ngày lookback. Để loại bỏ dữ liệu trùng lặp, có thể sử dụng hàm drop_duplicate trong Pandas.
3. Pipeline không thể biết được sự kiện xóa dòng trong bảng order_items vì hệ thống nhận diện sự thay đổi dữ liệu order_items thông qua watermark_column (updated_at), khi dữ liệu updated_at của bản ghi không còn do việc xóa bản ghi thì hệ thống cũng sẽ không nhận diện được bản ghi bị xóa, không thể dựa vào việc không ghi nhận dữ liệu qua các lần ingres để xác định là dữ liệu đó đã bị xóa vì các order_items không có sự cập nhật sau này cũng không được ghi nhận qua các lần ingres như các bản ghi thật sự bị xóa.